In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Track Gemini usage and cost per team on shared Provisioned Throughput

| Author |
| --- |
| [Kasin Sumalyaporn](https://github.com/kasin-gg) |

## Overview

When several teams share one Google Cloud project, Cloud Billing shows Gemini cost **per project and usage type**, not per team.
This is common with Provisioned Throughput, because an order and its fixed fee belong to one project.

This notebook shows how to attribute usage and cost to each team:

1. **Account identity:** each team calls Gemini with its own service account. The inference request metadata log records the account, token counts and Provisioned Throughput use of every request, even without labels. It never stores prompts or responses.
2. **Request labels:** optional business detail (team, app, cost center) that appears on cost lines in the Cloud Billing export.
3. **Monthly allocation:** split the Provisioned Throughput fixed fee by each team's share of reserved-capacity tokens, and charge overflow at its actual cost.

### Before you begin

Run the setup first, using either `scripts/01_setup.sh` or the Terraform module in `terraform/`. It creates one service account per team,
switches on inference request metadata logs, and copies them to a BigQuery dataset with a log sink.
Your user account needs the Service Account Token Creator role on the team accounts to send the sample requests.

## Get started

### Install the libraries

In [ ]:
%pip install --upgrade --quiet google-genai google-cloud-bigquery google-auth pandas db-dtypes

### Authenticate (Colab only)

In Vertex AI Workbench or Cloud Shell, your environment is already authenticated.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set your project and settings

In [ ]:
PROJECT_ID = "your-shared-project"  # @param {type: "string"}
LOCATION = "global"  # @param {type: "string"}
MODEL = "gemini-3.8-flash"  # @param {type: "string"}
TEAMS = ["team-a", "team-b"]
DATASET = "gemini_usage"  # @param {type: "string"}
USAGE_TABLE = "aiplatform_googleapis_com_inference_request"  # @param {type: "string"}
# Optional: billing-project.billing_dataset.gcp_billing_export_resource_v1_XXXXXX_XXXXXX_XXXXXX
BILLING_TABLE = ""  # @param {type: "string"}

### Import libraries and create clients

In [ ]:
import google.auth
from google.auth import impersonated_credentials
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from google import genai
from google.genai import types

SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
user_credentials, _ = google.auth.default(scopes=SCOPES)
bq = bigquery.Client(project=PROJECT_ID, credentials=user_credentials)
API_HOST = "aiplatform.googleapis.com" if LOCATION == "global" else f"{LOCATION}-aiplatform.googleapis.com"


def team_account(team: str) -> str:
    return f"{team}-sa@{PROJECT_ID}.iam.gserviceaccount.com"


def team_client(team: str) -> genai.Client:
    """A Gemini client that calls the API as the team's service account."""
    creds = impersonated_credentials.Credentials(
        source_credentials=user_credentials, target_principal=team_account(team), target_scopes=SCOPES
    )
    return genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, credentials=creds)

## Step 1: Check that inference request metadata logs are on

The setting is project-wide, so it covers every Gemini model, including ones you add later.

In [ ]:
session = AuthorizedSession(user_credentials)
config = session.get(f"https://aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}:fetchPublisherModelConfig").json()
print(config)
assert config.get("inferenceEventLoggingConfig", {}).get("state") == "ENABLED", \
    "Metadata logs are off. Run scripts/01_setup.sh or terraform apply."

## Step 2: Send requests as each team

Each team calls Gemini with its own service account. Half of the requests carry a `consumer` label and half do not,
to show that the metadata log attributes usage either way.

In [ ]:
rows = []
for team in TEAMS:
    client = team_client(team)
    for i in range(2):
        for labelled in (True, False):
            config = types.GenerateContentConfig(
                max_output_tokens=200, labels={"consumer": team} if labelled else None
            )
            response = client.models.generate_content(
                model=MODEL, contents=f"In about 60 words, explain cost attribution (sample {i}).", config=config
            )
            usage = response.usage_metadata
            rows.append({"team": team, "labelled": labelled, "total_tokens": usage.total_token_count,
                         "traffic_type": getattr(usage.traffic_type, "name", str(usage.traffic_type))})

import pandas as pd

pd.DataFrame(rows)

`traffic_type` is `ON_DEMAND` for pay-as-you-go requests. When a Provisioned Throughput order is active, requests served
from reserved capacity show `PROVISIONED_THROUGHPUT`.

## Step 3: Usage per team from the metadata log

The log sink creates the table with the first request it copies, and new rows usually appear within a few minutes. Rerun the cell if the table is still missing or empty.

In [ ]:
import time

def query(sql: str, **params) -> pd.DataFrame:
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter(k, "FLOAT64" if isinstance(v, float) else "DATE", v) for k, v in params.items()])
    return bq.query(sql, job_config=job_config).to_dataframe()

USAGE_SQL = f"""
SELECT e.principal.id AS account, e.traffictype AS traffic_type, COUNT(*) AS requests,
  CAST(SUM(e.generatecontentdetails.usagemetadata.totaltokencount) AS INT64) AS total_tokens,
  CAST(SUM(IFNULL(SAFE_CAST(JSON_VALUE(TO_JSON(e), '$.provisionedthroughput.ptchargedinputtokencount') AS FLOAT64), 0)
         + IFNULL(SAFE_CAST(JSON_VALUE(TO_JSON(e), '$.provisionedthroughput.ptchargedoutputtokencount') AS FLOAT64), 0))
       AS INT64) AS pt_charged_tokens
FROM `{PROJECT_ID}.{DATASET}.{USAGE_TABLE}`, UNNEST([jsonpayload_logging_inferenceeventlogentry]) AS e
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY) AND severity != 'ERROR'
GROUP BY account, traffic_type ORDER BY account"""

for attempt in range(10):
    try:
        usage = query(USAGE_SQL)
    except Exception as e:  # The table appears after the sink copies its first entry.
        print("Waiting for the table:", str(e)[:120])
        usage = pd.DataFrame()
    if not usage.empty:
        break
    time.sleep(30)
usage["share_pct"] = (100 * usage.total_tokens / usage.total_tokens.sum()).round(1)
usage

## Step 4: Cost per team label from the Cloud Billing export (optional)

The detailed billing export adds cost and labels to BigQuery, but new usage can take about 1.5 days to appear.
Requests sent without a label appear as `(no label)`, and label values are kept exactly as sent.

In [ ]:
if BILLING_TABLE:
    display(query(f"""
    SELECT IFNULL((SELECT value FROM UNNEST(labels) WHERE key = 'consumer'), '(no label)') AS team,
      SUM(usage.amount) AS tokens, ROUND(SUM(cost), 4) AS cost
    FROM `{BILLING_TABLE}`
    WHERE _PARTITIONTIME >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 35 DAY)
      AND project.id = '{PROJECT_ID}' AND sku.description LIKE 'Gemini%'
    GROUP BY team ORDER BY cost DESC"""))
else:
    print("Set BILLING_TABLE to check cost in the billing export.")

## Step 5: Split the Provisioned Throughput fixed fee

The fee is split by each team's share of the Provisioned Throughput burndown (`pt_charged_tokens`), which the log records on every request served from reserved capacity. Overflow is billed separately
at pay-as-you-go prices, so charge it from the billing export. Before an order is active, this example uses all tokens instead,
so you can see the calculation.

In [ ]:
FIXED_FEE = 10000.0  # Illustrative. Use your contracted monthly fee.
reserved = usage[usage.traffic_type == "PROVISIONED_THROUGHPUT"]
basis = reserved if not reserved.empty else usage
if reserved.empty:
    print("No PROVISIONED_THROUGHPUT rows yet, so this example uses all tokens.")
col = "pt_charged_tokens" if not reserved.empty else "total_tokens"
alloc = basis.groupby("account", as_index=False)[col].sum()
alloc["share_pct"] = (100 * alloc[col] / alloc[col].sum()).round(1)
alloc["fixed_fee_share"] = (FIXED_FEE * alloc[col] / alloc[col].sum()).round(2)
alloc

## Clean up

To remove the setup, run `scripts/99_cleanup.sh` or `terraform destroy`. Deleting the dataset deletes the logs permanently.